***Data Loading and Exploration:***

In [10]:
import pandas as pd
import numpy as np

In [11]:
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

In [12]:
fake_df.head()
fake_df.shape
fake_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [13]:
true_df.head()
true_df.shape
true_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [14]:
fake_df['label'] = 0  #Fake
true_df['label'] = 1 #True

In [15]:
df = pd.concat([true_df, fake_df], axis=0)

In [16]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
df.head()
df.shape
df.info()
df['label'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


label
0    23481
1    21417
Name: count, dtype: int64

In [21]:
df['content'] = df['title'] + " " + df['text']

***Text Preprocessing and NLP Cleaning:***

In [22]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [23]:
def basic_clean(text):
    text = text.lower()                    #lowercasing
    text = re.sub(r'[^a-z\s]', '', text)   #remove punctuation and numbers
    text = re.sub(r'\s+', '', text).strip() #remoce extra spaces
    return text

In [24]:
df['clean_text'] = df['content'].apply(basic_clean)

In [27]:
def remove_stopwords(text):
    tokens = text.split()  #tokenization
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return "".join(tokens)

In [28]:
df['clean_text'] = df['clean_text'].apply(remove_stopwords)

In [29]:
df[['content', 'clean_text']].head()

,content,clean_text
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,breakinggopchairmangrassleyhashadenoughdemands...
1,Failed GOP Candidates Remembered In Hilarious...,failedgopcandidatesrememberedinhilariousmockin...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,mikepencesnewdcneighborsarehilariouslytrolling...
3,California AG pledges to defend birth control ...,californiaagpledgestodefendbirthcontrolinsuran...
4,AZ RANCHERS Living On US-Mexico Border Destroy...,azrancherslivingonusmexicoborderdestroynancype...


***Feature Extraction:***

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
tfidf = TfidfVectorizer(
    max_features = 5000,  #limit voab size
    ngram_range = (1, 2)  #unigrams + bigrams and captures word pairs like “fake news”
)

In [34]:
X = df['clean_text']   #x=input text
y = df['label']        #y=target(0=fake, 1=real)    

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y         #Ensures fake & real proportions remain balanced
)

In [36]:
X_train_tfidf = tfidf.fit_transform(X_train)    #fit_transform → training data
X_test_tfidf = tfidf.transform(X_test)          #transform → test data

In [37]:
X_train_tfidf.shape

(35918, 5000)

***Model Training:***

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report 
)

In [40]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [41]:
y_pred_lr = log_reg.predict(X_test_tfidf)

In [42]:
print("Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1-Score:", f1_score(y_test, y_pred_lr))

Logistic Regression Results:
Accuracy: 0.5025612472160357
Precision: 0.48954405210833046
Recall: 1.0
F1-Score: 0.657307249712313


In [43]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       1.00      0.05      0.09      4696
           1       0.49      1.00      0.66      4284

    accuracy                           0.50      8980
   macro avg       0.74      0.52      0.38      8980
weighted avg       0.76      0.50      0.36      8980



In [44]:
confusion_matrix(y_test, y_pred_lr)

array([[ 229, 4467],
       [   0, 4284]])

***Save Trained Model & TF-IDF Vectorizer:***

In [45]:
import joblib

In [46]:
import os

os.makedirs("models", exist_ok=True)

In [48]:
joblib.dump(log_reg, "models/fake_news_model.pkl")


['models/fake_news_model.pkl']

In [49]:
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

['models/tfidf_vectorizer.pkl']

In [50]:
os.listdir("models")

['fake-news-model.pkl', 'fake_news_model.pkl', 'tfidf_vectorizer.pkl']

In [51]:
loaded_model = joblib.load("models/fake_news_model.pkl")
loaded_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")

In [52]:
sample_text = ["this is a fake news example text"]
sample_tfidf = loaded_vectorizer.transform(sample_text)
prediction = loaded_model.predict(sample_tfidf)

print(prediction)

[1]


***Sentiment Analysis (Post-Model Insight)***

In [61]:
positive_words = {
    "good", "great", "positive", "success", "benefit", "win", "growth",
    "improve", "safe", "true", "trusted", "official", "confirmed",
    "support", "approved", "peace", "hope", "progress"
}

negative_words = {
     "bad", "fake", "false", "fraud", "hoax", "lie", "scam",
    "danger", "fear", "hate", "loss", "wrong", "misleading",
    "attack", "crisis", "threat", "kill", "destroy", "violence",
    "corrupt", "illegal", "shock", "outrage"
}

In [62]:
def get_sentiment_sample(text):
    words = text.split()

    pos_count = sum(1 for word in words if word in positive_words)
    neg_count = sum(1 for word in words if word in negative_words)

    if pos_count > neg_count:
        return "Postive"
    elif neg_count > pos_count:
        return "Negative"
    else:
        return "Neutral"

In [63]:
df['sentiment'] = df['clean_text'].apply(get_sentiment_sample)


In [64]:
df[['clean_text', 'sentiment']].head()       #Check
                                             

,clean_text,sentiment
0,breakinggopchairmangrassleyhashadenoughdemands...,Neutral
1,failedgopcandidatesrememberedinhilariousmockin...,Neutral
2,mikepencesnewdcneighborsarehilariouslytrolling...,Neutral
3,californiaagpledgestodefendbirthcontrolinsuran...,Neutral
4,azrancherslivingonusmexicoborderdestroynancype...,Neutral


In [65]:
sentiment_distribution = df.groupby(['label', 'sentiment']).size()
sentiment_distribution

label  sentiment
0      Neutral      23481
1      Neutral      21417
dtype: int64

In [66]:
sentiment_distribution.unstack()


sentiment,Neutral
label,
0,23481
1,21417


**Observation:
Lexicon-based sentiment analysis classified the majority of articles as neutral. This aligns with the objective tone commonly used in news reporting. Hence, sentiment analysis was treated as an exploratory insight rather than a predictive feature.**